# IPL Performance Intelligence

## Project Overview

The Indian Premier League (IPL) is one of the world's most competitive T20 cricket leagues. Teams invest heavily in player acquisitions, coaching staff, and match strategies to gain a competitive advantage.

This project aims to analyze historical IPL match and ball-by-ball data to identify factors that contribute to winning matches. The analysis focuses on team performance, toss decisions, venue characteristics, and scoring patterns.

The ultimate goal is to uncover actionable insights that can help explain match outcomes and strategic advantages across different IPL seasons.

## Objectives

This analysis seeks to answer the following questions:

- Does winning the toss improve the probability of winning a match?
- Is it better to bat first or chase a target?
- Which teams perform best under different match conditions?
- Which venues consistently favor batting or bowling?
- How important are powerplay and death-over performances?
- What factors have the strongest influence on match outcomes?

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Loading the Datasets

Two datasets are used in this project:

### Matches Dataset
Contains match-level information such as teams, venue, toss results, match winners, and season details.

### Deliveries Dataset
Contains ball-by-ball records for every IPL match, allowing detailed analysis of scoring patterns, wickets, and team strategies.

In [3]:
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

In [4]:
print("Matches Shape:", matches.shape)
print("Deliveries Shape:", deliveries.shape)

Matches Shape: (1095, 20)
Deliveries Shape: (260920, 17)


## Dataset Size

Before proceeding with the analysis, it is important to understand the scale of the available data.

The matches dataset provides match-level information across multiple IPL seasons, while the deliveries dataset records every ball bowled during those matches.

A larger dataset increases confidence in the reliability of the insights generated during later stages of analysis.

In [5]:
matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


## Initial Review of Match-Level Data

The matches dataset contains information related to participating teams, venues, toss outcomes, match winners, and season details.

Understanding the structure of the data helps identify which variables can be used for performance analysis and strategic insights.

In [6]:
deliveries.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


## Initial Review of Ball-by-Ball Data

The deliveries dataset captures every delivery bowled in IPL history.

This level of detail enables advanced analysis such as:

- Powerplay performance
- Death-over scoring
- Wicket patterns
- Run-rate trends
- Team scoring efficiency

Ball-by-ball information is particularly valuable when investigating the factors that contribute to winning matches.

In [8]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

In [9]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          260920 non-null  int64 
 1   inning            260920 non-null  int64 
 2   batting_team      260920 non-null  object
 3   bowling_team      260920 non-null  object
 4   over              260920 non-null  int64 
 5   ball              260920 non-null  int64 
 6   batter            260920 non-null  object
 7   bowler            260920 non-null  object
 8   non_striker       260920 non-null  object
 9   batsman_runs      260920 non-null  int64 
 10  extra_runs        260920 non-null  int64 
 11  total_runs        260920 non-null  int64 
 12  extras_type       14125 non-null   object
 13  is_wicket         260920 non-null  int64 
 14  player_dismissed  12950 non-null   object
 15  dismissal_kind    12950 non-null   object
 16  fielder           9354 non-null    obj

## Data Structure Assessment

A review of data types and available fields helps determine:

- Which variables require cleaning
- Whether any columns contain missing values
- Opportunities for feature engineering
- Suitability of the data for further analysis

Understanding data quality at an early stage reduces the likelihood of inaccurate conclusions later in the project.

In [10]:
matches.isnull().sum()

id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

In [11]:
deliveries.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

## Missing Value Analysis

Missing values are a common challenge in real-world datasets.

Before performing any analysis, it is important to identify incomplete records and determine whether they require treatment, removal, or business interpretation.

The findings from this step will guide the data-cleaning process in the next phase of the project.

## Key Observations from Initial Data Review

### Match Dataset

- The dataset contains 1,095 IPL matches across multiple seasons.
- Match-level information includes teams, venue, toss outcomes, match results, and player awards.
- Most columns have complete data with relatively few missing values.

### Missing Values Identified

- `city` contains 51 missing records.
- `player_of_match` contains 5 missing records.
- `winner` contains 5 missing records.
- `result_margin` contains 19 missing records.
- `target_runs` and `target_overs` contain 3 missing records each.
- `method` contains a large number of missing values, indicating that special match outcome methods were rarely used.

### Deliveries Dataset

- The dataset contains 260,920 ball-by-ball records.
- No missing values were found in core scoring columns.
- Missing values in dismissal-related columns are expected because most deliveries do not result in wickets.

### Initial Assessment

The overall quality of the data appears strong and suitable for further analysis.

Most missing values appear to be business-related rather than data-quality issues. For example, dismissal information is naturally unavailable when no wicket occurs.

The next phase will focus on data cleaning, consistency checks, and preparation for deeper performance analysis.